# Dracula `death_toll` — parametric-knowledge probe

Does a model already know the answer, or does it need the corpus?

Two conditions per model, on the same question:

Both are the **`direct-llm`** baseline — one LLM call, no retrieval. Only the setup
changes, recorded as `condition` in the run config:

| condition | prompt |
|---|---|
| **parametric** | the question alone — no documents |
| **with-corpus** | all 46 documents in the system prompt, `direct-llm` framing |

Everything lands in `logs/dracula/direct-llm/`; the two conditions form separate
analysis groups because `condition` is part of the config.

Runs write **standard harness manifests**, so `python -m evals.analysis --benchmark dracula
--baseline direct-llm` grades them with the same dual strict/lenient v9 judge as every
other baseline, and the numbers are directly comparable.

Resumes: a `(model, seed, condition)` already on disk is skipped.


In [ ]:
# The notebook lives inside the package, so put the repo root on sys.path.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("repo root:", _root)

In [ ]:
import json, uuid
from pathlib import Path

from evals.baselines import _common
from evals.benchmarks import dracula
from evals.llm.chat import litellm_chat_completion_full
from evals.llm.usage import usage_envelope

TASK_ID = "death_toll"
RUN_VERSION = "v1"

# direct-llm's own framing, so the with-corpus arm is what direct-llm would do.
INSTRUCTION = "Answer the question based on the following text. Answer concisely."

QUESTION, DOCS = dracula.get_task(TASK_ID)
print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS):,} chars")
print("question:", QUESTION)

In [ ]:
# Names the novel explicitly in the parametric arm. This is deliberate and makes the
# probe HARDER to dismiss: the model is told exactly which text the question is about,
# so a correct answer cannot be explained away as it having misread the question. The
# with-corpus arm never sees this — there the corpus is the source of truth.
PARAMETRIC_FRAMING = "In Bram Stoker's 1897 novel Dracula, I have the following question:"


def _build_prompts(condition: str):
    """(system_prompt, user_prompt) for a condition.

    parametric  — no documents at all; the model answers from its priors, but IS told
                  which novel the question refers to (see PARAMETRIC_FRAMING).
    with-corpus — every document in the system prompt under direct-llm's framing
                  (`## Document N`), the bare question as the user message.
    """
    if condition == "parametric":
        return "", f"{PARAMETRIC_FRAMING}\n\n{QUESTION}"
    body = "\n\n".join(f"## Document {i+1}\n\n{d}" for i, d in enumerate(DOCS))
    return f"{INSTRUCTION}\n\n{body}", QUESTION


BASELINE = "direct-llm"          # both conditions ARE direct-llm; only the setup differs


def run_probe(model, condition, seed=42, *, base_url=None, api_key=None, **call_kwargs):
    """One probe → one standard manifest under logs/dracula/<baseline>/<run_tag>/.

    Skips if a manifest for this exact config already exists (resumption).
    """
    base = _common.base_dir("dracula", BASELINE)
    config = {
        "benchmark": "dracula",
        "baseline": BASELINE,
        "model": _common.canonical_model_id(model),
        "seed": seed,
        "condition": condition,
        "run_version": RUN_VERSION,
    }
    if condition == "parametric":
        # The framing is part of what the parametric arm measures, so it belongs in the
        # run identity: changing it must not silently reuse a run made under the old one.
        config["framing"] = "names-the-novel"
    if TASK_ID in _common.scan_completed_task_ids(base, config):
        print(f"  · skip (done)  {config['model']:<20} {condition}  seed={seed}")
        return None

    system_prompt, user_prompt = _build_prompts(condition)
    response = litellm_chat_completion_full(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        model=_common.with_provider_prefix(model),
        api_base=base_url,
        api_key=api_key,
        seed=seed,
        **call_kwargs,
    )
    raw_answer = response.choices[0].message.content or ""

    run_dir = base / uuid.uuid4().hex[:12]
    _common.write_manifest(run_dir, {
        "task_id": TASK_ID,
        "config": config,
        "usage": usage_envelope(response),       # deterministic, single call
        "raw_answer": raw_answer,
        "trace": {
            "condition": condition,
            "n_docs": 0 if condition == "parametric" else len(DOCS),
            "system_prompt_chars": len(system_prompt),
            "user_prompt_chars": len(user_prompt),
            "question": QUESTION,
        },
    })
    tok = sum(v.get("total_tokens", 0) for v in usage_envelope(response)["total"].values())
    print(f"  ✓ {config['model']:<20} {condition:<12} seed={seed}  {tok:>7,} tok  → {run_dir.name}")
    return raw_answer

## Configure

`MODELS` maps a model id to the kwargs it needs. OpenAI models need nothing extra; the
served Qwen needs `base_url` + `api_key`. Add seeds to `SEEDS` for a noise band.

In [ ]:
MODELS = {
    "openai/gpt-6-astra": {},
    "Qwen/Qwen3.5-35B-A3B": {
        "base_url": "http://localhost:8555/v1",
        "api_key": "your_secret",
    },
}

SEEDS = [42]                                  # e.g. [42, 1, 2, 3, 4] for a noise band
CONDITIONS = ["parametric", "with-corpus"]

In [ ]:
for model, kw in MODELS.items():
    for condition in CONDITIONS:
        for seed in SEEDS:
            try:
                run_probe(model, condition, seed, **kw)
            except Exception as exc:
                print(f"  ✗ {model} {condition} seed={seed}: {type(exc).__name__}: {exc}")

## Score and compare

Grades every probe with dracula's dual judge (strict = the 13-victim roster, lenient =
Swales optional) and caches a `score.json` beside each manifest, exactly like the other
baselines. Re-running is free.

In [ ]:
from evals.analysis.aggregate import find_groups

for grp in find_groups(benchmark="dracula", baseline=BASELINE):
    cfg = grp["config"]
    print(f"{cfg.get('model'):<20} {cfg.get('condition'):<12} seed={cfg.get('seed')}  "
          f"{len(grp['manifests'])} manifest(s), {len(grp['errors'])} error(s)")

In [ ]:
# Grade every probe with dracula's dual judge and tabulate.
from pathlib import Path
from evals.analysis.score import ensure_scores, read_score
from evals.benchmarks.dracula.judge import read_verdicts

groups = find_groups(benchmark="dracula", baseline=BASELINE)
infs = [(Path(m["_inference_dir"]), m) for g in groups for m in g["manifests"]]
rows = []
if infs:
    stats = ensure_scores(infs, dracula)
    print(f"graded {stats['scored']}, cached {stats['cached']}, total {stats['total']}\n")
    for d, m in infs:
        s = read_score(d) or {}
        v = read_verdicts(s.get("parsed"))
        rows.append({
            "model": m["config"]["model"],
            "condition": m["config"].get("condition", "?"),
            "seed": m["config"].get("seed"),
            "strict": v[0] if v else s.get("score"),
            "lenient": v[1] if v else None,
            "answer": " ".join(m["raw_answer"].split()),
        })

print(f"{'model':<20} {'condition':<12} {'seed':>4} {'strict':>7} {'lenient':>8}  answer")
for r in sorted(rows, key=lambda r: (r["model"], r["condition"], r["seed"] or 0)):
    print(f"{r['model']:<20} {r['condition']:<12} {str(r['seed']):>4} "
          f"{str(r['strict']):>7} {str(r['lenient']):>8}  {r['answer'][:70]}")

## Read the answers in full

In [ ]:
base = _common.base_dir("dracula", BASELINE)
for d in sorted(base.iterdir()):
    mf = d / "manifest.json"
    if not mf.exists():
        continue
    m = json.loads(mf.read_text())
    c = m["config"]
    print("=" * 100)
    print(f"{c['model']}  ·  {c.get('condition')}  ·  seed {c.get('seed')}  ·  {d.name}")
    print("=" * 100)
    print(m["raw_answer"])
    print()